In [8]:

!pip install -q segmentation-models-pytorch opencv-python matplotlib pandas

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import segmentation_models_pytorch as smp

from google.colab import drive
drive.mount('/content/drive')


FOLDER = "/content/drive/MyDrive/Colab Notebooks/google_earth_test"
MODEL_PATH = "/content/drive/MyDrive/Colab Notebooks/best_ayat_working_model.pth"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
THRESHOLD = 0.30

print("DEVICE:", DEVICE)


model = smp.UnetPlusPlus(
    encoder_name="efficientnet-b3",
    encoder_weights=None,
    in_channels=17,
    classes=1,
    activation=None
).to(DEVICE)

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

print("MODEL LOADED SUCCESSFULLY")


def safe_div(a, b):
    return a / (b + 1e-6)

def make_17_channels(img):
    img = cv2.resize(img, (256, 256))
    img = img.astype(np.float32) / 255.0

    R = img[:, :, 0]
    G = img[:, :, 1]
    B = img[:, :, 2]

    gray = (R + G + B) / 3.0

    red_ratio   = safe_div(R, G)
    green_ratio = safe_div(G, B)
    blue_ratio  = safe_div(B, R)
    ndvi_like   = safe_div(G - R, G + R)

    grad_x = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    gradient = np.sqrt(grad_x**2 + grad_y**2)

    var_gray = cv2.blur(gray**2, (7,7)) - cv2.blur(gray, (7,7))**2

    x = np.stack([
        R, G, B,
        gray,
        red_ratio,
        green_ratio,
        blue_ratio,
        ndvi_like,
        gradient,
        var_gray,
        R, G, B,
        gray,
        gradient,
        var_gray,
        gray
    ], axis=0).astype(np.float32)


    for i in range(x.shape[0]):
        c = x[i]
        x[i] = (c - c.min()) / (c.max() - c.min() + 1e-6)

    return x

files = [f for f in os.listdir(FOLDER)
         if f.lower().endswith(('.png','.jpg','.jpeg'))]

files.sort()

print("FOUND FILES:", files)


results = []

for file in files:

    path = os.path.join(FOLDER, file)

    img = cv2.imread(path)

    if img is None:
        continue

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    x = make_17_channels(img)

    tensor = torch.tensor(x, dtype=torch.float32).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred = model(tensor)

    prob = torch.sigmoid(pred)[0,0].cpu().numpy()
    mask = (prob > THRESHOLD).astype(np.uint8)


    predicted_area_percent = mask.mean() * 100
    mean_probability = prob.mean()
    max_probability = prob.max()
    confidence_score = mean_probability * 100

    results.append({
        "filename": file,
        "predicted_area_%": round(predicted_area_percent, 2),
        "mean_probability": round(mean_probability, 4),
        "max_probability": round(max_probability, 4),
        "confidence_%": round(confidence_score, 2)
    })


    ys, xs = np.where(mask == 1)

    step = max(1, len(xs) // 300)

    xs_sample = xs[::step]
    ys_sample = ys[::step]


    plt.figure(figsize=(24,6))

    plt.subplot(1,4,1)
    plt.imshow(img)
    plt.title(file)
    plt.axis("off")

    plt.subplot(1,4,2)
    plt.imshow(prob, cmap="gray")
    plt.title("Probability Map")
    plt.axis("off")

    plt.subplot(1,4,3)
    plt.imshow(mask, cmap="gray")
    plt.title("Predicted Mask")
    plt.axis("off")

    plt.subplot(1,4,4)
    plt.imshow(cv2.resize(img, (256,256)))
    plt.scatter(xs_sample, ys_sample, c="red", marker="^", s=25)
    plt.title("Detected Zones")
    plt.axis("off")

    plt.show()


df = pd.DataFrame(results)

df = df.sort_values("confidence_%", ascending=False)

print("\nFINAL RESULTS TABLE:")
print(df)


csv_path = "/content/google_earth_results.csv"
df.to_csv(csv_path, index=False)

print("\nCSV SAVED:", csv_path)

Output hidden; open in https://colab.research.google.com to view.